# Read the Iceberg lakehouse table

Connect to the Spark Connect server at `sc://localhost:15002`. 

The table is `polaris.telemetry.gps_processed` is resolved inside Spark. 

#### 1. Connect to Spark

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
print(f"Connected to Spark {spark.version} at sc://localhost:15002")

Connected to Spark 4.0.1 at sc://localhost:15002


#### 2. Analyze latest data

In [6]:
spark.sql("""
    SELECT
        CAST(FLOOR(speed_kmh / 10) * 10 AS INT) AS speed_bin_kmh,
        COUNT(*) AS message_count,
        ROUND(
            100.0 * SUM(CASE WHEN is_valid = false THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS invalid_events_percent
    FROM polaris.telemetry.gps_processed
    WHERE kafka_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 5 MINUTES
      AND speed_kmh IS NOT NULL
    GROUP BY CAST(FLOOR(speed_kmh / 10) * 10 AS INT)
    ORDER BY speed_bin_kmh
""").show(truncate=False)

+-------------+-------------+----------------------+
|speed_bin_kmh|message_count|invalid_events_percent|
+-------------+-------------+----------------------+
|10           |17023        |6.74                  |
|20           |16999        |6.61                  |
|30           |17358        |6.46                  |
|40           |16896        |6.56                  |
|50           |17105        |6.95                  |
|60           |86           |6.98                  |
+-------------+-------------+----------------------+

